# 第 17 章 · Hopper ①：TMA、mbarrier、cluster 与分布式 shared memory

本章回答：`cp.async` 要每个线程各搬 16 字节，搬一个 128 × 64 的 tile 要算一堆地址；Hopper 的 TMA 怎么做到"一个线程一条指令搬整块"？mbarrier 怎么"按字节数"判断数据到齐？thread block cluster 让 block 之间能做什么以前不能做的事？

配套讲义：仓库里 `cuda-programming/notes/lessons/17-hopper-tma-mbarrier.md`（讲解、动画、练习都在讲义里）。对应原文：PG §4.12.2 Using TMA（`materials/text/cuda-programming-guide.txt` 第 19711–20750 行）、§4.10 Asynchronous Barriers（第 17711–18583 行）、§1.2.2.1.1 Thread Block Clusters（第 1059–1076 行）、§2.1.10（第 2557–2600 行）、§2.3.3.8 Distributed Shared Memory（第 3573–3742 行）；Colfax「Mastering the NVIDIA TMA」（`text/blog-colfax-hopper-tma.txt`）；PTX §9.7.10.28.5.5 cp.async.bulk.tensor（`text/ptx-isa-9.4-focus-mma-wgmma-tcgen05-tma.txt` 第 3605 行）、§9.7.15.16 mbarrier（`text/ptx-isa-9.4.txt` 第 24642 行）、§5.5 Tensors（`text/ptx-isa-9.4.txt` 第 3020 行）；Hopper tuning guide §4.1.2–4.1.3

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU：代码用 NumPy 在 CPU 上模拟 GPU 的规则（线程编号、32 字节一箱、32 个 bank……）。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 tensor map | 按坐标搬 tile | `TensorMap(A, (4, 4)).load_tile(x, y)` | 5 × 6 矩阵的 tile (1, 1) 只有 29、30，其余补 0 |
| 第二步 16 字节规则 | 行 stride % 16 == 0 | `stride_ok(n_cols, 4)` | FP32 列数 8：True；6：False |
| 第三步 mbarrier | arrive.expect_tx + TMA 记账 | `MBarrier(1)`、`arrive_expect_tx(16384)`、`complete_tx(16384)` | 搬完前 False，搬完后 True，相位 0 → 1 |
| 第四步 DSMEM | `map_shared_rank(buf, rank ^ 1)` | `cluster_exchange(bufs)` | 两个 block 各读到对方的数 |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**类 `MBarrier`**（来自 `ch17_tma_mbarrier.py`）：shared memory 里的 64 位对象：同时记"还差几次 arrive"和"还差多少字节"。两个都清零 → 相位翻转、重置。

In [ ]:
class MBarrier:
    """shared memory 里的 64 位对象：同时记"还差几次 arrive"和"还差多少字节"。两个都清零 → 相位翻转、重置。"""

    def __init__(self, arrive_count: int):
        self.expected = arrive_count
        self.pending_arrive = arrive_count
        self.pending_tx = 0
        self.phase = 0

    def _maybe_flip(self):
        if self.pending_arrive == 0 and self.pending_tx == 0:
            self.phase ^= 1
            self.pending_arrive = self.expected

    def arrive_expect_tx(self, nbytes: int):
        """mbarrier.arrive.expect_tx：报到一次，并声明"还要来 nbytes 字节"。"""
        self.pending_tx += nbytes
        self.pending_arrive -= 1
        self._maybe_flip()

    def complete_tx(self, nbytes: int):
        """TMA 搬完一部分时硬件自动调用：扣掉相应字节。"""
        self.pending_tx -= nbytes
        self._maybe_flip()

    def try_wait(self, parity: int) -> bool:
        """mbarrier.try_wait.parity：parity 这一相位已经完成了吗。"""
        return self.phase != parity

**类 `TensorMap`**（来自 `ch17_tma_mbarrier.py`）：host 端写好的"仓库地图"：全局张量的形状 + tile（box）大小。这里只做二维、行优先。

In [ ]:
class TensorMap:
    """host 端写好的"仓库地图"：全局张量的形状 + tile（box）大小。这里只做二维、行优先。"""

    def __init__(self, tensor: np.ndarray, box: tuple[int, int]):
        self.tensor = tensor
        self.box = box

    def load_tile(self, x: int, y: int) -> np.ndarray:
        """cp.async.bulk.tensor.2d：把坐标 (x, y) 的 tile（第 x 块行、第 y 块列）搬出来；越界部分填 0。"""
        bh, bw = self.box
        out = np.zeros(self.box, dtype=self.tensor.dtype)
        part = self.tensor[x * bh:(x + 1) * bh, y * bw:(y + 1) * bw]   # 越界的部分 NumPy 切片自动截掉
        out[:part.shape[0], :part.shape[1]] = part
        return out

**函数 `cluster_exchange`**（来自 `ch17_tma_mbarrier.py`）：cluster 大小 2：block rank r 用 map_shared_rank(buf, r ^ 1) 读伙伴的 shared memory（DSMEM）。

In [ ]:
def cluster_exchange(bufs: list[np.ndarray]) -> list[np.ndarray]:
    """cluster 大小 2：block rank r 用 map_shared_rank(buf, r ^ 1) 读伙伴的 shared memory（DSMEM）。"""
    return [bufs[r ^ 1].copy() for r in range(len(bufs))]

**函数 `stride_ok`**（来自 `ch17_tma_mbarrier.py`）：TMA 要求：除最内层外的每个 stride 是 16 字节的倍数。行优先矩阵的行 stride = 列数 × 元素字节。

In [ ]:
def stride_ok(n_cols: int, elem_bytes: int) -> bool:
    """TMA 要求：除最内层外的每个 stride 是 16 字节的倍数。行优先矩阵的行 stride = 列数 × 元素字节。"""
    return n_cols * elem_bytes % 16 == 0

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 tensor map：5 × 6 的矩阵，tile 4 × 4")
A = np.arange(1, 31).reshape(5, 6)
tm = TensorMap(A, (4, 4))
print("  tile (0, 0)：\n", tm.load_tile(0, 0))
print("  tile (1, 1)（大半越界，自动补 0）：\n", tm.load_tile(1, 1))
print("  FP32 矩阵列数 8：", stride_ok(8, 4), "；列数 6：", stride_ok(6, 4), "（要先 padding）")

In [ ]:
print("\n第三步 mbarrier：期望 1 次 arrive，TMA 搬 16384 字节")
bar = MBarrier(1)
bar.arrive_expect_tx(16384)
print("  报到后、TMA 还没搬完：能过吗？", bar.try_wait(0), "，还差", bar.pending_tx, "字节")
bar.complete_tx(16384)
print("  TMA 搬完：能过吗？", bar.try_wait(0), "，相位翻成", bar.phase)
bar.arrive_expect_tx(16384)
print("  下一圈用 parity 1 等：", bar.try_wait(1), "（这一圈的数据还没到）")

In [ ]:
print("\n第四步 cluster 里两个 block 互读 shared memory")
bufs = [np.full(4, 10.0), np.full(4, 20.0)]
print("  block 0 读到伙伴的：", cluster_exchange(bufs)[0], "；block 1 读到：", cluster_exchange(bufs)[1])

## 3. 代码里才看得到的

1. **tensor map 的坐标是"第几块"**：`load_tile(1, 1)` 取的是第 1 块行、第 1 块列，即原矩阵第 4–7 行、第 4–7 列；真实 TMA 的坐标是元素坐标，CuTe 会替你换算。
2. **越界补 0 靠 NumPy 切片自动截断**：`tensor[4:8, 4:8]` 在 5 × 6 的矩阵上只返回 1 × 2，剩下的位置保持 0——TMA 在硬件里做同样的事，kernel 不用写 `if`。
3. **mbarrier 要人头和字节都清零才翻相位**：只报到不等字节、或者字节到了没人报到，都过不去。

## 4. 自己改一改

- 把 `TensorMap` 的 box 改成 (2, 3)，打印 tile (2, 1)，看越界部分。
- 把 `MBarrier(1)` 改成 `MBarrier(2)`，只调一次 `arrive_expect_tx`，看 TMA 搬完后能不能过。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> TMA 是专职搬运工：一张仓库地图（tensor map），一个线程一条指令搬整块，越界自动补零；
> mbarrier 按人头也按字节记账：expect_tx 先报数，搬完自动扣，清零翻相位；
> cluster 把相邻 block 绑一组：同时上岗、能互相等、能翻对方的台面（DSMEM）；
> multicast 一次读、几家同时收。